# Lesson 6: Behavior

**You will learn:** how to describe a task with `TaskContext`, get answers through a readout, and read implied probabilities and reaction times.
**Requires:** CPU only, under a minute.

A `TaskContext` says what the subject is asked to do; its `response` says what form the answer takes: `Choice(labels)`, `Scale(...)` for ratings, or `Text()`. A network trained for something else answers through a **readout**: a classifier fitted on one layer with labeled examples. Its probabilities and reaction times are **implied** by its activity; a model that replies in words gives **reported** answers instead (see [Experiment 14](../experiments/14_reading_instructions_and_words.ipynb)).

**The question:** is the readout less certain on the images it gets wrong? Its implied reaction time (`evidence_margin`: 1 minus the gap between the two most probable answers) should then be longer on errors.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

In [5]:
# Half the images (6 per category) fit readouts; the other half test them.
order = stimuli.groupby('category').cumcount()
train = StimulusSet(stimuli[order < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[order >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

## The task

In [6]:
from brainscore_core import TaskContext
from brainscore_core.responses import Choice
from brainscore_core.streaming import BufferedSession

task = TaskContext(response=Choice(['animal', 'vehicle'], probabilities=True, reaction_time='evidence_margin'),
                   fitting_stimuli=train, prefer_path='readout')

def answers(s):
    session = BufferedSession(stimulus_set=test, task_context=task,
                              requested_output_channels=['behavior:choice', 'behavior:probabilities', 'behavior:rt'])
    s.participate(session)
    s.reset()
    return {c.split(':')[1]: session.collect(c) for c in session.requested_output_channels}

result = answers(subject)
chosen = np.array(['animal', 'vehicle'])[result['choice'].values.argmax(1)]
correct = chosen == np.asarray(test['label'])
print('held-out accuracy:', round(correct.mean(), 3), ' measure:', result['rt'].attrs['measure'])

held-out accuracy: 0.833  measure: {'kind': 'implied', 'link': 'evidence_margin', 'source': 'readout'}


## The null and the question

In [7]:
torch.manual_seed(0)
untrained = squeezenet_subject(torchvision.models.squeezenet1_1().eval(), 'squeezenet1_1-untrained')
null = answers(untrained)
null_correct = np.array(['animal', 'vehicle'])[null['choice'].values.argmax(1)] == np.asarray(test['label'])
print('untrained network, held-out accuracy:', round(null_correct.mean(), 3))

from scipy.stats import mannwhitneyu
rt = np.asarray(result['rt'].values)
print(f'implied RT: correct {rt[correct].mean():.3f} (n={correct.sum()}), errors {rt[~correct].mean():.3f} (n={(~correct).sum()})')
if (~correct).any():
    print('errors slower: p =', round(mannwhitneyu(rt[~correct], rt[correct], alternative='greater').pvalue, 4))

untrained network, held-out accuracy: 0.767
implied RT: correct 0.172 (n=50), errors 0.364 (n=10)
errors slower: p = 0.3717


Read the p-value with the number of errors beside it: with ten or so errors on 60 test images, even a large difference in means may not be reliable, and the answer is to test more images, not to trust the means. A readout that is reliably slower on its errors behaves like a participant whose hard trials are the ones they get wrong; a reaction-time benchmark would test whether its slow images are people's slow images.

## Summary

- `TaskContext(response=Choice(...), fitting_stimuli=...)` describes the task; held-out stimuli test it.
- Readouts give implied probabilities and reaction times, labelled in `attrs['measure']`.
- The untrained network is the null for what the trained one answers.

**Next:** [Lesson 7](07_perturb_a_model.ipynb) changes the network during a run.